# 金融テキストからのKG構築
## Knowledge Graph Construction from Financial Text

---

### 背景

金融テキスト（ニュース記事、有価証券報告書、SEC提出書類（10-K、10-Q）、決算説明会議事録など）には、
企業間の関係、人物の役職、買収、業績データなど、膨大な構造化可能な情報が含まれている。
これらの非構造化テキストから **知識グラフ（Knowledge Graph, KG）** を自動構築することは、
金融分析・リスク管理・投資意思決定を高度化するための重要な研究課題である。

### パイプライン概要

本ノートブックでは、以下のパイプラインに沿って金融テキストからKGを構築する：

1. **固有表現認識（Named Entity Recognition, NER）** — テキストから企業名、人物名、金額、日付、地名等のエンティティを抽出
2. **関係抽出（Relation Extraction）** — エンティティ間の関係（買収、役職、所在地、競合等）をルールベースで抽出
3. **RDFグラフ構築** — 抽出したトリプル（主語-述語-目的語）をRDF形式で格納
4. **可視化・クエリ** — 構築したKGをグラフとして可視化し、SPARQLで問い合わせ

### 関連ドキュメント

本ノートブックの理論的背景については、以下のドキュメントを参照されたい：

- [金融ネットワークとナレッジグラフに対する計算機科学の視点（03-computer-science.md）](../03-computer-science.md)
  - 特に「金融ナレッジグラフプロジェクト」セクション（FinDKG、FinKG等）
  - NLPパイプライン（NER＋関係抽出）からKG構築へのアーキテクチャ

---
## 1. 環境セットアップ

必要なライブラリをインストールする。

In [ ]:
# 必要なライブラリのインストール
!pip install spacy rdflib matplotlib networkx

# spaCyの英語モデルをダウンロード
!python -m spacy download en_core_web_sm

---
## 2. ライブラリのインポート

In [ ]:
import spacy
from spacy import displacy
from rdflib import Graph, Namespace, Literal, URIRef, RDF, RDFS
import networkx as nx
import matplotlib.pyplot as plt
import re
import json

# spaCyの英語モデルをロード
nlp = spacy.load("en_core_web_sm")

print("ライブラリのインポートが完了しました。")
print(f"spaCyバージョン: {spacy.__version__}")

---
## 3. サンプル金融テキストの準備

SEC EDGAR の 10-K（年次報告書）風のサンプルテキストを用意する。
実際の10-Kファイリングから抽出される典型的な情報（売上高、買収、役員、競合企業、所在地等）を含む。

In [ ]:
# SEC EDGAR 10-K風のサンプルテキスト
# 実際の提出書類を模した英語テキストを複数段落用意する

sample_texts = [
    # Apple Inc. に関するテキスト
    (
        "Apple Inc. reported revenue of $394.3 billion for fiscal year 2022. "
        "The company acquired Beats Electronics in 2014 for approximately $3 billion. "
        "Tim Cook serves as Chief Executive Officer. "
        "Apple's main competitors include Samsung Electronics, Google LLC, and Microsoft Corporation. "
        "The company is headquartered in Cupertino, California."
    ),
    # JPMorgan Chase に関するテキスト
    (
        "JPMorgan Chase & Co. reported revenue of $128.7 billion for fiscal year 2022. "
        "The firm acquired Bear Stearns in 2008 for approximately $1.2 billion. "
        "Jamie Dimon serves as Chairman and Chief Executive Officer. "
        "JPMorgan Chase's main competitors include Bank of America, Citigroup, and Goldman Sachs. "
        "The company is headquartered in New York City, New York."
    ),
    # Tesla Inc. に関するテキスト
    (
        "Tesla Inc. reported revenue of $81.5 billion for fiscal year 2022. "
        "The company acquired SolarCity in 2016 for approximately $2.6 billion. "
        "Elon Musk serves as Chief Executive Officer. "
        "Tesla's main competitors include Toyota Motor Corporation, Volkswagen AG, and BYD Company. "
        "The company is headquartered in Austin, Texas."
    ),
]

# 全テキストを結合して表示
full_text = " ".join(sample_texts)

print("=== サンプル金融テキスト ===")
for i, text in enumerate(sample_texts, 1):
    print(f"\n--- 段落 {i} ---")
    print(text)

---
## 4. spaCy による固有表現抽出（NER）

spaCyの `en_core_web_sm` モデルを使用して、テキストから固有表現（Named Entity）を抽出する。
金融テキストにおいて重要なエンティティタイプ：

| ラベル | 説明 | 例 |
|--------|------|----|
| `ORG` | 組織・企業 | Apple Inc., JPMorgan Chase |
| `PERSON` | 人物 | Tim Cook, Jamie Dimon |
| `MONEY` | 金額 | $394.3 billion |
| `DATE` | 日付・期間 | fiscal year 2022 |
| `GPE` | 地名（国、都市等） | Cupertino, California |

In [ ]:
# 各段落に対してNERを実行
docs = [nlp(text) for text in sample_texts]

# 抽出されたエンティティを一覧表示
print("=== 固有表現抽出（NER）結果 ===")
for i, doc in enumerate(docs, 1):
    print(f"\n--- 段落 {i} ---")
    for ent in doc.ents:
        print(f"  テキスト: {ent.text:<30} ラベル: {ent.label_:<10} 説明: {spacy.explain(ent.label_)}")

# エンティティタイプ別に集計
print("\n=== エンティティタイプ別集計 ===")
entity_counts = {}
for doc in docs:
    for ent in doc.ents:
        label = ent.label_
        entity_counts[label] = entity_counts.get(label, 0) + 1

for label, count in sorted(entity_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {label}: {count}件")

---
## 5. NER結果の可視化

`spacy.displacy` を使用して、NER結果をハイライト表示する。
各エンティティが色分けされ、視覚的に確認できる。

In [ ]:
# displacy を使ってNER結果を可視化（Jupyter上でHTML表示）
# 各段落のNER結果をレンダリング
for i, doc in enumerate(docs, 1):
    print(f"--- 段落 {i} のNER可視化 ---")
    displacy.render(doc, style="ent", jupyter=True)

---
## 6. ルールベースの関係抽出

NERで抽出したエンティティ間の関係を、正規表現によるパターンマッチングで抽出する。
本ノートブックでは、金融テキストに頻出する以下の5つの関係パターンを対象とする：

| パターン | 関係 | 例 |
|----------|------|----|
| `acquired` | 買収関係 | (Apple, acquired, Beats Electronics) |
| `serves as` | 役職関係 | (Tim Cook, hasRole, CEO) |
| `headquartered in` | 所在地関係 | (Apple, locatedIn, Cupertino) |
| `competitors include` | 競合関係 | (Apple, competitorOf, Samsung) |
| `reported revenue of` | 売上高関係 | (Apple, hasRevenue, $394.3 billion) |

In [ ]:
def extract_relations(texts):
    """
    ルールベースのパターンマッチングにより、金融テキストから関係トリプルを抽出する。
    
    戻り値: (主語, 述語, 目的語) のタプルのリスト
    """
    triples = []
    
    for text in texts:
        # パターン1: 買収関係 — "X acquired Y in YEAR for AMOUNT"
        pattern_acquired = r"([A-Z][\w\s&.]+?)\s+acquired\s+([A-Z][\w\s]+?)\s+in\s+(\d{4})"
        for match in re.finditer(pattern_acquired, text):
            acquirer = match.group(1).strip()
            target = match.group(2).strip()
            year = match.group(3).strip()
            triples.append((acquirer, "acquired", target))
            triples.append((acquirer, "acquisitionYear", year))
        
        # パターン2: 役職関係 — "PERSON serves as ROLE"
        pattern_role = r"([A-Z][a-z]+\s+[A-Z][a-z]+)\s+serves\s+as\s+(.+?)\."
        for match in re.finditer(pattern_role, text):
            person = match.group(1).strip()
            role = match.group(2).strip()
            triples.append((person, "hasRole", role))
        
        # パターン3: 所在地関係 — "headquartered in LOCATION"
        pattern_hq = r"([A-Z][\w\s&.]+?)\s+is\s+headquartered\s+in\s+([A-Z][\w\s,]+?)\."
        for match in re.finditer(pattern_hq, text):
            company = match.group(1).strip()
            # "The company" を直前の企業名に置き換える
            if company == "The company":
                # テキスト冒頭の企業名を取得
                org_match = re.match(r"([A-Z][\w\s&.]+?)\s+reported", text)
                if org_match:
                    company = org_match.group(1).strip()
            location = match.group(2).strip()
            triples.append((company, "locatedIn", location))
        
        # パターン4: 競合関係 — "X's main competitors include A, B, and C"
        pattern_comp = r"([A-Z][\w\s&.]+?)'s\s+main\s+competitors\s+include\s+(.+?)\."
        for match in re.finditer(pattern_comp, text):
            company = match.group(1).strip()
            competitors_str = match.group(2).strip()
            # カンマと "and" で分割
            competitors = re.split(r",\s*(?:and\s+)?", competitors_str)
            competitors = [c.replace("and ", "").strip() for c in competitors]
            for comp in competitors:
                if comp:
                    triples.append((company, "competitorOf", comp))
        
        # パターン5: 売上高関係 — "X reported revenue of AMOUNT"
        pattern_rev = r"([A-Z][\w\s&.]+?)\s+reported\s+revenue\s+of\s+(\$[\d.]+\s+billion)"
        for match in re.finditer(pattern_rev, text):
            company = match.group(1).strip()
            revenue = match.group(2).strip()
            triples.append((company, "hasRevenue", revenue))
    
    return triples


# 関係抽出を実行
extracted_triples = extract_relations(sample_texts)

# 抽出結果を表示
print("=== 抽出された関係トリプル ===")
print(f"合計: {len(extracted_triples)}件\n")

for i, (subj, pred, obj) in enumerate(extracted_triples, 1):
    print(f"  {i:2d}. ({subj}, {pred}, {obj})")

---
## 7. RDFグラフの構築

抽出したトリプルを **RDF（Resource Description Framework）** グラフに格納する。
RDFは、知識グラフの標準的な表現形式であり、主語-述語-目的語のトリプルでデータを記述する。

ここでは `rdflib` を使用して、独自の名前空間 `ex:` を定義し、トリプルをグラフに追加する。

In [ ]:
# RDFグラフの初期化
rdf_graph = Graph()

# 名前空間の定義
EX = Namespace("http://example.org/finance/")
rdf_graph.bind("ex", EX)
rdf_graph.bind("rdfs", RDFS)

def sanitize_uri(name):
    """
    エンティティ名をURI用の安全な文字列に変換する。
    空白をアンダースコアに置換し、特殊文字を除去する。
    """
    # 空白をアンダースコアに変換
    sanitized = name.replace(" ", "_")
    # URIに使えない文字を除去
    sanitized = re.sub(r"[^\w_.-]", "", sanitized)
    return sanitized


# 抽出したトリプルをRDFグラフに追加
for subj, pred, obj in extracted_triples:
    subj_uri = EX[sanitize_uri(subj)]
    pred_uri = EX[sanitize_uri(pred)]
    
    # 金額や年などのリテラル値はLiteralとして格納
    if pred in ["hasRevenue", "acquisitionYear"]:
        rdf_graph.add((subj_uri, pred_uri, Literal(obj)))
    else:
        obj_uri = EX[sanitize_uri(obj)]
        rdf_graph.add((subj_uri, pred_uri, obj_uri))
    
    # エンティティにRDFSラベルを付与（可読性向上のため）
    rdf_graph.add((subj_uri, RDFS.label, Literal(subj)))
    if pred not in ["hasRevenue", "acquisitionYear"]:
        rdf_graph.add((EX[sanitize_uri(obj)], RDFS.label, Literal(obj)))

print(f"RDFグラフに {len(rdf_graph)} 個のトリプルを格納しました。")

---
## 8. RDFグラフの出力（Turtle形式）

構築したRDFグラフを **Turtle形式** で出力する。
Turtle形式はRDFの可読性の高いシリアライゼーション形式であり、
トリプルの構造を直感的に把握できる。

In [ ]:
# Turtle形式でRDFグラフをシリアライズ
turtle_output = rdf_graph.serialize(format="turtle")

print("=== RDFグラフ（Turtle形式） ===")
print(turtle_output)

---
## 9. networkx による知識グラフ可視化

RDFトリプルを `networkx` のグラフに変換し、`matplotlib` で可視化する。
ノードはエンティティ、エッジは関係を表し、エッジラベルに関係名を表示する。

In [ ]:
def build_networkx_graph(rdf_graph, namespace):
    """
    RDFグラフからnetworkxの有向グラフを構築する。
    RDFSラベルは除外し、実質的な関係のみをエッジとして追加する。
    """
    G = nx.DiGraph()
    
    for subj, pred, obj in rdf_graph:
        # RDFSラベルのトリプルはスキップ（可視化対象外）
        if pred == RDFS.label:
            continue
        
        # URIからローカル名を取得（表示用）
        subj_name = str(subj).replace(str(namespace), "").replace("_", " ")
        pred_name = str(pred).replace(str(namespace), "").replace("_", " ")
        
        if isinstance(obj, Literal):
            obj_name = str(obj)
        else:
            obj_name = str(obj).replace(str(namespace), "").replace("_", " ")
        
        G.add_edge(subj_name, obj_name, label=pred_name)
    
    return G


# networkxグラフを構築
G = build_networkx_graph(rdf_graph, EX)

print(f"ノード数: {G.number_of_nodes()}")
print(f"エッジ数: {G.number_of_edges()}")

# 可視化
fig, ax = plt.subplots(1, 1, figsize=(18, 14))

# レイアウトの計算（spring layout で見やすく配置）
pos = nx.spring_layout(G, k=2.5, iterations=50, seed=42)

# ノードの描画
nx.draw_networkx_nodes(G, pos, node_color="lightblue", node_size=2000, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight="bold", ax=ax)

# エッジの描画
nx.draw_networkx_edges(G, pos, edge_color="gray", arrows=True,
                       arrowsize=20, connectionstyle="arc3,rad=0.1", ax=ax)

# エッジラベル（関係名）の描画
edge_labels = nx.get_edge_attributes(G, "label")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                             font_size=7, font_color="red", ax=ax)

ax.set_title("金融知識グラフ（Knowledge Graph）の可視化", fontsize=16)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 10. SPARQLクエリ

`rdflib` の SPARQL クエリ機能を使って、構築した知識グラフに問い合わせを行う。
SPARQLは、RDFデータに対する標準的なクエリ言語であり、
複雑なグラフパターンの検索が可能である。

In [ ]:
# クエリ1: Apple と競合関係にある企業は？
print("=== クエリ1: Apple Inc. と競合関係にある企業 ===")

query1 = """
PREFIX ex: <http://example.org/finance/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?competitor ?label
WHERE {
    ex:Apple ?pred ?competitor .
    FILTER(CONTAINS(STR(?pred), "competitorOf"))
    OPTIONAL { ?competitor rdfs:label ?label . }
}
"""

results1 = rdf_graph.query(query1)
for row in results1:
    label = row.label if row.label else row.competitor
    print(f"  競合企業: {label}")

# クエリ2: 各企業のCEOは誰か？
print("\n=== クエリ2: 各企業の CEO ===")

query2 = """
PREFIX ex: <http://example.org/finance/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?person ?personLabel ?role
WHERE {
    ?person ex:hasRole ?role .
    OPTIONAL { ?person rdfs:label ?personLabel . }
    FILTER(CONTAINS(STR(?role), "Chief Executive Officer"))
}
"""

results2 = rdf_graph.query(query2)
for row in results2:
    label = row.personLabel if row.personLabel else row.person
    print(f"  人物: {label} — 役職: {row.role}")

# クエリ3: 全ての買収関係を取得
print("\n=== クエリ3: 全ての買収関係 ===")

query3 = """
PREFIX ex: <http://example.org/finance/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?acquirer ?acquirerLabel ?target ?targetLabel
WHERE {
    ?acquirer ex:acquired ?target .
    OPTIONAL { ?acquirer rdfs:label ?acquirerLabel . }
    OPTIONAL { ?target rdfs:label ?targetLabel . }
}
"""

results3 = rdf_graph.query(query3)
for row in results3:
    acq_label = row.acquirerLabel if row.acquirerLabel else row.acquirer
    tgt_label = row.targetLabel if row.targetLabel else row.target
    print(f"  {acq_label} が {tgt_label} を買収")

# クエリ4: 各企業の売上高
print("\n=== クエリ4: 各企業の売上高 ===")

query4 = """
PREFIX ex: <http://example.org/finance/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?company ?companyLabel ?revenue
WHERE {
    ?company ex:hasRevenue ?revenue .
    OPTIONAL { ?company rdfs:label ?companyLabel . }
}
ORDER BY DESC(?revenue)
"""

results4 = rdf_graph.query(query4)
for row in results4:
    label = row.companyLabel if row.companyLabel else row.company
    print(f"  {label}: {row.revenue}")

---
## 11. 考察

### ルールベース手法の限界

本ノートブックで実装したルールベースの関係抽出は、以下の限界がある：

- **パターンの網羅性**: 正規表現パターンは手動で定義する必要があり、テキストの表現揺れ（例："acquired" vs. "purchased" vs. "bought"）に対応しきれない。
- **文脈の欠如**: 単純なパターンマッチングでは、文脈に依存する関係（暗黙的な関係、否定表現等）を捉えられない。
- **照応解析の不足**: 「The company」「The firm」などの代名詞・指示表現の解決が不十分。
- **スケーラビリティ**: 新しい関係タイプを追加するたびにパターンの追加が必要。

### LLMを使った関係抽出への発展

近年、大規模言語モデル（LLM）を活用した関係抽出が急速に進展している：

- **プロンプトベースの抽出**: GPT-4やClaude等のLLMに対して、テキストから関係トリプルを直接抽出するようプロンプトを設計する手法。
- **FinDKG**: Imperial College London のプロジェクトでは、LLMを使って金融ニュースから動的KGを構築している（[03-computer-science.md](../03-computer-science.md) 参照）。
- **FinReflectKG**: LLMによるKG構築と反省的改善を組み合わせた手法。

### 金融オントロジーとの統合

実用的な金融KGの構築には、標準的なオントロジーとの統合が不可欠：

- **FIBO（Financial Industry Business Ontology）**: EDM Council が管理する金融業界標準オントロジー。企業、金融商品、契約等の概念を体系的に定義。
- **GLEIF（Global Legal Entity Identifier Foundation）**: 法人識別子（LEI）を用いたグローバルなエンティティ解決。
- **Schema.org**: 汎用的な構造化データの語彙（Organization, Person 等）。

### 大規模KGの構築と維持の課題

- **データ品質**: 自動抽出されたトリプルの精度と再現率のトレードオフ。人手によるキュレーションのコスト。
- **時間的変化**: 金融情報は時間とともに変化するため、KGの継続的な更新が必要（動的KG）。
- **エンティティ解決**: 異なるソース間での同一エンティティの統合（例："Apple" = "Apple Inc." = "AAPL"）。
- **スケーラビリティ**: 数百万ノード規模のKGを効率的に格納・クエリするためのグラフデータベース（Neo4j, Amazon Neptune 等）の活用。